# **[Phase 2] Real-World Field Evaluation & Matcher Comparison (nantah-memorial-P42)**

### **Experiment Objective & Strategy**

##### **The Objective: Real-World Field Evaluation**

In Phase 1 of this project, the primary objective was to strictly validate the mathematical core of the Incremental Structure from Motion (SfM) pipeline within a highly controlled environment. Now in Phase 2, the objective shifts to evaluating the pipeline's robustness against unconstrained, real-world data collection. By introducing custom-captured datasets that feature complex environmental factors (such as repetitive architectural patterns, textureless surfaces, background foliage clutter, and steep perspective distortions), we can aggressively stress-test the feature tracking and geometric failsafes (like global relocalization) of our baseline architecture.

##### **The Field Evaluation Strategy & nantah-memorial-P42**

To conduct this real-world evaluation, this project employs custom datasets captured on the Nanyang Technological University (NTU) campus. Specifically, the experiments focus on two distinct architectural subjects: the Nanyang University Memorial (Nantah Memorial) and the heritage red pavilions located in Yunnan Garden.

This specific notebook focuses on the first environment: **nantah-memorial-P42**. Featuring a sequence of 42 high-resolution images orbiting the monument, it serves as a rigorous test for out-of-plane affine distortions. The memorial's design, characterized by highly repetitive architectural patterns and large, uniform concrete surfaces, is intentionally selected to challenge traditional feature extractors and Lowe's ratio test logic. This dataset directly exposes the limitations of hand-crafted gradients when confronted with architectural symmetry and textureless structural facades.

##### **Feature Matcher Evaluation**

During this field evaluation phase, the core engine is executed iteratively to compare how three distinct feature extraction architectures handle real-world scene ambiguity:
* **SIFT (Scale-Invariant Feature Transform):** Evaluated specifically for its vulnerability to repetitive textures, foliage, and rapid perspective distortion on unconstrained structures.
* **SuperPoint + LightGlue:** An advanced, PyTorch-driven deep learning framework integrated to evaluate how Transformer-based cross-attention and semantic context can overcome the failures of traditional matchers on symmetric and textureless targets.
* **ORB (Oriented FAST and Rotated BRIEF):** A lightweight, binary-descriptor alternative utilized to test if aggressive downscaling and high-speed corner detection can extract stable geometry from high-frequency natural noise.

##### **Core Engine Architecture: `incremental-sfm`**

To execute this validation phase, the notebook continues to rely on the [`chuo-v/incremental-sfm`](https://github.com/chuo-v/incremental-sfm) repository, which serves as the core codebase for the project. This repository encapsulates the computational heavy lifting of the pipeline, housing the pure Python implementations for feature extraction, outlier rejection, global relocalization, and multi-view triangulation. By keeping this central geometry engine strictly decoupled from the datasets and analytical notebooks, the exact same mathematical primitives that successfully cleared the Phase 1 benchmarks can be seamlessly applied to these challenging NTU field captures.

### **Infrastructure & Environment Setup**

In [1]:
import os
import sys

In [2]:
CORE_REPO_OWNER = "chuo-v"
CORE_REPO_NAME = "incremental-sfm"
CORE_TAG_NAME = "v1.0.0"

if 'google.colab' in str(get_ipython()):
    from google.colab import userdata

    if not os.path.exists(CORE_REPO_NAME):
        try:
            token = userdata.get('GITHUB_TOKEN')
            repo_url = f"https://{token}@github.com/{CORE_REPO_OWNER}/{CORE_REPO_NAME}.git"
            print(f"Cloning private repository ({CORE_TAG_NAME}) via PAT...")
        except userdata.SecretNotFoundError:
            repo_url = f"https://github.com/{CORE_REPO_OWNER}/{CORE_REPO_NAME}.git"
            print(f"Cloning public repository ({CORE_TAG_NAME})...")

        !git clone -q -b {CORE_TAG_NAME} --depth 1 {repo_url} &> /dev/null

    os.chdir(CORE_REPO_NAME)
    print(f"Environment Ready: Working directory set to {os.getcwd()}")

else:
    # Local Execution: Assume incremental-sfm is cloned as a sibling directory
    local_repo_path = os.path.abspath(f"../{CORE_REPO_NAME}")

    if os.path.exists(local_repo_path):
        os.chdir(local_repo_path)
        print(f"Environment Ready: Working directory set to {os.getcwd()}")
    else:
        print(f"Error: {CORE_REPO_NAME} not found at {local_repo_path}.")
        print("Please clone the core engine repository as a sibling to your current folder.")

Cloning private repository (v1.0.0) via PAT...
Environment Ready: Working directory set to /content/incremental-sfm


In [3]:
%%capture
!pip install -e .

### **Data Preparation & Camera Intrinsics**

In [4]:
from pathlib import Path

In [5]:
data_dir = Path("../datasets").resolve()
dataset_name = "nantah-memorial-P42"

os.environ["DATA_DIR"] = str(data_dir)
os.environ["DATASET"] = dataset_name

with open(".env", "w") as f:
    f.write(f"DATA_DIR={data_dir}\n")

print(f"Configured DATA_DIR={data_dir} for dataset={dataset_name}")

Configured DATA_DIR=/content/datasets for dataset=nantah-memorial-P42


In [6]:
DATASET_TARGET_DIR = data_dir / dataset_name
DATASET_REPO_OWNER = "chuo-v"
DATASET_REPO_NAME = "incremental-sfm-datasets"
DATASET_TAG_NAME = "v1.0.0"

if not DATASET_TARGET_DIR.exists():
    import shutil
    from google.colab import userdata

    print(f"Downloading {dataset_name} dataset via sparse checkout ({DATASET_TAG_NAME})...")

    try:
        token = userdata.get('GITHUB_TOKEN')
        repo_url = f"https://{token}@github.com/{DATASET_REPO_OWNER}/{DATASET_REPO_NAME}.git"
    except userdata.SecretNotFoundError:
        print("GITHUB_TOKEN not found. Attempting public clone...")
        repo_url = f"https://github.com/{DATASET_REPO_OWNER}/{DATASET_REPO_NAME}.git"

    !git clone -q -b {DATASET_TAG_NAME} --depth 1 --filter=blob:none --sparse {repo_url} temp_datasets_repo &> /dev/null
    !cd temp_datasets_repo && git sparse-checkout set {dataset_name} &> /dev/null

    DATASET_TARGET_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(f"temp_datasets_repo/{dataset_name}", str(DATASET_TARGET_DIR))
    shutil.rmtree("temp_datasets_repo")

    print(f"Dataset ready at: {DATASET_TARGET_DIR}")
else:
    print(f"Dataset {dataset_name} already exists at {DATASET_TARGET_DIR}.")

Dataset ready at: /content/datasets/nantah-memorial-P42


### **Pipeline Execution**

##### **SIFT Execution**

Executes the pipeline using SIFT (Scale-Invariant Feature Transform) to establish a baseline for the limitations of traditional computer vision in unconstrained environments. For this dataset, SIFT is specifically evaluated on its vulnerability to the memorial's repetitive architectural symmetry and smooth, textureless concrete surfaces, as well as the severe out-of-plane affine distortion introduced as the camera rapidly orbits the structure.

In [7]:
!python main.py \
    --dataset {dataset_name} \
    --matcher SIFT \
    --downscale_factor 0.5 \
    --max_cache_size 100 \
    --max_features 50000 \
    --lowe_ratio_threshold 0.80 \
    --apply_clahe \
    --ransac_prob 0.999 \
    --ransac_threshold 6.0 \
    --min_parallax_degrees 1.0

2026-10-05 23:44:55,820 - INFO - ==============================
2026-10-05 23:44:55,820 - INFO - Starting Incremental SfM Pipeline
2026-10-05 23:44:55,821 - INFO - Dataset: nantah-memorial-P42
2026-10-05 23:44:55,821 - INFO - Matcher: SIFT
2026-10-05 23:44:55,821 - INFO - Data Directory: /content/datasets
2026-10-05 23:44:55,821 - INFO - ==============================
2026-10-05 23:44:55,838 - INFO - Initializing Two-View Seed (Images 0 and 1)...
2026-10-05 23:47:42,792 - INFO - Two-View Initialization complete (166.953s). Seeded 2458 3D points.
2026-10-05 23:47:42,792 - INFO - Processing Image 2 / 41...
2026-10-05 23:51:06,825 - INFO - Registered Image 2 (204.033s): 1127 PnP Inliers | 1569 Triangulated Points
2026-10-05 23:51:06,825 - INFO - Processing Image 3 / 41...
2026-10-05 23:55:02,933 - INFO - Registered Image 3 (236.107s): 1367 PnP Inliers | 2357 Triangulated Points
2026-10-05 23:55:02,933 - INFO - Processing Image 4 / 41...
2026-10-05 23:58:44,109 - INFO - Registered Image 4 

##### **SuperPoint Execution**

Integrates PyTorch-driven SuperPoint and LightGlue to evaluate the resilience of modern deep learning architectures against real-world scene ambiguity. This execution tests the ability of Transformer-based cross-attention and semantic context to successfully maintain sequential tracking and dense 3D mapping across the steep perspective changes and repetitive decorative patterns that cause traditional hand-crafted descriptors to fail.

In [8]:
!python main.py \
    --dataset {dataset_name} \
    --matcher SuperPoint \
    --downscale_factor 0.5 \
    --max_cache_size 100 \
    --max_features 50000 \
    --lowe_ratio_threshold 0.80 \
    --apply_clahe \
    --ransac_prob 0.999 \
    --ransac_threshold 6.0 \
    --min_parallax_degrees 1.0

2026-10-06 01:33:54,720 - INFO - ==============================
2026-10-06 01:33:54,720 - INFO - Starting Incremental SfM Pipeline
2026-10-06 01:33:54,720 - INFO - Dataset: nantah-memorial-P42
2026-10-06 01:33:54,720 - INFO - Matcher: SuperPoint
2026-10-06 01:33:54,720 - INFO - Data Directory: /content/datasets
2026-10-06 01:33:54,720 - INFO - ==============================
2026-10-06 01:33:54,723 - INFO - Initializing SuperPoint & LightGlue on device: cpu
Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_v1.pth" to /root/.cache/torch/hub/checkpoints/superpoint_v1.pth
100% 4.96M/4.96M [00:00<00:00, 86.7MB/s]
Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_lightglue.pth" to /root/.cache/torch/hub/checkpoints/superpoint_lightglue_v0-1_arxiv.pth
100% 45.3M/45.3M [00:00<00:00, 166MB/s]
2026-10-06 01:33:55,970 - INFO - Initializing Two-View Seed (Images 0 and 1)...
2026-10-06 01:35:00,593 - INFO - Two-View Initializ

##### **ORB Execution**

Utilizes the lightweight ORB binary descriptor to evaluate the pipeline's high-speed computational efficiency on challenging field data. This execution measures whether fast corner detection and aggressive downscaling can extract stable geometry from high-frequency environmental noise like background foliage, or if the algorithm collapses under the monument's rapid out-of-plane rotations and lack of distinct micro-texture.

In [9]:
!python main.py \
    --dataset {dataset_name} \
    --matcher ORB \
    --downscale_factor 0.5 \
    --max_cache_size 100 \
    --max_features 50000 \
    --lowe_ratio_threshold 0.80 \
    --apply_clahe \
    --ransac_prob 0.999 \
    --ransac_threshold 6.0 \
    --min_parallax_degrees 1.0

2026-10-06 02:05:59,299 - INFO - ==============================
2026-10-06 02:05:59,299 - INFO - Starting Incremental SfM Pipeline
2026-10-06 02:05:59,299 - INFO - Dataset: nantah-memorial-P42
2026-10-06 02:05:59,299 - INFO - Matcher: ORB
2026-10-06 02:05:59,299 - INFO - Data Directory: /content/datasets
2026-10-06 02:05:59,299 - INFO - ==============================
2026-10-06 02:05:59,303 - INFO - Initializing Two-View Seed (Images 0 and 1)...
2026-10-06 02:07:23,406 - INFO - Two-View Initialization complete (84.104s). Seeded 1252 3D points.
2026-10-06 02:07:23,407 - INFO - Processing Image 2 / 41...
2026-10-06 02:08:45,961 - INFO - Registered Image 2 (82.555s): 348 PnP Inliers | 806 Triangulated Points
2026-10-06 02:08:45,962 - INFO - Processing Image 3 / 41...
2026-10-06 02:10:09,074 - INFO - Registered Image 3 (83.113s): 294 PnP Inliers | 952 Triangulated Points
2026-10-06 02:10:09,074 - INFO - Processing Image 4 / 41...
2026-10-06 02:11:31,529 - INFO - Registered Image 4 (82.455s

### **Quantitative Metrics**

This section aggregates and visualizes the JSON output metrics generated by each feature matcher. By comparing point cloud density, Mean Reprojection Error (RMSE), and average processing time side-by-side, we can quantitatively evaluate the engineering trade-offs between traditional descriptor limitations on unconstrained architecture (SIFT), AI-driven semantic resilience (SuperPoint), and baseline computational efficiency (ORB).

In [10]:
%%capture
!pip install kaleido==0.2.1 plotly==5.24.1

In [11]:
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [12]:
matchers = ["SIFT", "SuperPoint", "ORB"]
metrics_data = []

for matcher in matchers:
    metrics_path = data_dir / dataset_name / "output" / "metrics" / f"run_metrics_{matcher}.json"

    if metrics_path.exists():
        with open(metrics_path, "r") as f:
            data = json.load(f)
            summary = data["results"]["summary"]
            metrics_data.append({
                "Matcher": matcher,
                "Total 3D Points": summary.get("total_3d_points", 0),
                "Overall RMSE (px)": round(summary.get("overall_rmse", 0.0), 3),
                "Avg Frame Time (s)": round(summary.get("avg_frame_time_seconds", 0.0), 3)
            })
    else:
        print(f"Metrics not found for {matcher} at {metrics_path}")

In [13]:
df = pd.DataFrame(metrics_data)
print(f"Pipeline Summary ({dataset_name}):\n")
print(df.to_string(index=False))

Pipeline Summary (nantah-memorial-P42):

   Matcher  Total 3D Points  Overall RMSE (px)  Avg Frame Time (s)
      SIFT            52059              4.495             159.119
SuperPoint            41838              4.118              46.166
       ORB            38870              3.562              82.169


In [14]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Total 3D Points (Higher = Denser)", "Overall RMSE (Lower = Better)", "Avg Frame Time (Lower = Faster)")
)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, row in df.iterrows():
    fig.add_trace(go.Bar(x=[row["Matcher"]], y=[row["Total 3D Points"]], marker_color=colors[i], name=row["Matcher"], showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=[row["Matcher"]], y=[row["Overall RMSE (px)"]], marker_color=colors[i], showlegend=False), row=1, col=2)
    fig.add_trace(go.Bar(x=[row["Matcher"]], y=[row["Avg Frame Time (s)"]], marker_color=colors[i], showlegend=False), row=1, col=3)

fig.update_layout(title_text=f"Quantitative Comparison: {dataset_name}", height=450, template="plotly_white")
fig.show()

# Export figure
vis_dir = data_dir / dataset_name / "output" / "visualizations"
vis_dir.mkdir(parents=True, exist_ok=True)
bar_chart_path = vis_dir / f"metrics_bar_chart_{dataset_name}.png"
fig.write_image(str(bar_chart_path), scale=2)
print(f"\nBar chart exported to: {bar_chart_path}")


Bar chart exported to: /content/datasets/nantah-memorial-P42/output/visualizations/metrics_bar_chart_nantah-memorial-P42.png


In [15]:
max_pts = df["Total 3D Points"].max() if not df.empty else 1

# Calculate dynamic padding for the y-axis
y_min = df["Overall RMSE (px)"].min() if not df.empty else 0
y_max = df["Overall RMSE (px)"].max() if not df.empty else 1
y_pad = (y_max - y_min) * 0.25 if y_max > y_min else 0.5

fig_tradeoff = go.Figure()

for i, row in df.iterrows():
    matcher = row["Matcher"]

    fig_tradeoff.add_trace(go.Scatter(
        x=[row["Avg Frame Time (s)"]],
        y=[row["Overall RMSE (px)"]],
        mode='markers+text',
        name=matcher,
        text=[f"<b>{matcher}</b><br>{row['Total 3D Points']} pts"],
        textposition="top center",
        cliponaxis=False,
        marker=dict(
            size=[row["Total 3D Points"]],
            sizemode='area',
            sizeref=2.0 * max_pts / (50.0 ** 2) if max_pts > 0 else 1,
            sizemin=10,
            color=colors[i],
            line=dict(width=2, color='DarkSlateGrey')
        )
    ))

fig_tradeoff.update_layout(
    title=f"Algorithmic Trade-Offs: Compute Time vs. Accuracy ({dataset_name})",
    xaxis_title="Average Frame Time (s)  ➔  [Lower is Faster]",
    yaxis_title="Overall RMSE (px)  ➔  [Lower is More Accurate]",
    yaxis=dict(range=[y_min - (y_pad * 0.5), y_max + y_pad]),
    template="plotly_white",
    height=500,
    showlegend=False,
    margin=dict(t=80, b=60, l=60, r=60)
)

fig_tradeoff.show()

# Export figure
tradeoff_chart_path = vis_dir / f"metrics_tradeoff_chart_{dataset_name}.png"
fig_tradeoff.write_image(str(tradeoff_chart_path), scale=2)
print(f"\nTrade-off chart exported to: {tradeoff_chart_path}")


Trade-off chart exported to: /content/datasets/nantah-memorial-P42/output/visualizations/metrics_tradeoff_chart_nantah-memorial-P42.png


### **3D Visualization & Filtering**

This section loads the raw 3D point clouds and applies statistical and radius-based outlier removal via Open3D to eliminate noise and spurious geometry. The cleaned point clouds are then rendered as full-width, interactive 3D plots using Plotly, allowing for a direct qualitative assessment of each feature matcher's structural accuracy and visual completeness.

In [16]:
%%capture
!pip install open3d

In [17]:
import open3d as o3d
import numpy as np

In [18]:
def visualize_point_cloud(
    dataset_name: str,
    matcher: str,
    nb_neighbors: int = 20,
    std_ratio: float = 2.0,
    nb_points: int = 3,
    radius: float = 1.0
):
    """Loads, filters, renders, and exports a full-width interactive 3D point cloud."""
    pc_dir = data_dir / dataset_name / "output" / "point_clouds"
    vis_dir = data_dir / dataset_name / "output" / "visualizations"
    vis_dir.mkdir(parents=True, exist_ok=True)

    ply_path = pc_dir / f"point_cloud_{matcher}.ply"

    if not ply_path.exists():
        print(f"[{matcher}] PLY file not found: {ply_path}")
        return

    print(f"[{matcher}] Loading point cloud from {ply_path}...")
    pcd = o3d.io.read_point_cloud(str(ply_path))

    if pcd.is_empty():
        print(f"[{matcher}] Point cloud is empty.")
        return

    original_count = len(pcd.points)

    # Apply statistical and radius outlier removal
    print(f"[{matcher}] Running outlier removal...")
    pcd_filtered, _ = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
    pcd_filtered, _ = pcd_filtered.remove_radius_outlier(nb_points=nb_points, radius=radius)

    filtered_count = len(pcd_filtered.points)
    print(f"[{matcher}] Filtered {original_count - filtered_count} extreme outliers. Remaining: {filtered_count} pts.")

    # Extract coordinates and standard 8-bit RGB colors
    points = np.asarray(pcd_filtered.points)
    colors_normalized = np.asarray(pcd_filtered.colors)
    colors_rgb = (colors_normalized * 255).astype(int)
    colors_str = [f'rgb({r}, {g}, {b})' for r, g, b in colors_rgb]

    # Render full-width interactive 3D plot
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode='markers',
        marker=dict(size=2, color=colors_str, opacity=0.8),
        name=matcher
    )])
    fig.update_layout(
        title=f"{matcher} Reconstruction ({dataset_name}) - {filtered_count} pts",
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data',
            camera=dict(up=dict(x=0, y=0, z=1), center=dict(x=0, y=0, z=0), eye=dict(x=0, y=-1.5, z=1.5))
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        height=800
    )

    # Export figure
    html_path = vis_dir / f"point_cloud_{matcher}_visualization.html"
    fig.write_html(str(html_path))
    print(f"[{matcher}] Interactive 3D plot exported to: {html_path}")

    # Open the interactive plot
    fig.show()

##### **SIFT**

In [19]:
visualize_point_cloud(
    dataset_name=dataset_name,
    matcher="SIFT",
    nb_neighbors=50,
    std_ratio=1.0,
    nb_points=200,
    radius=1.0
)

[SIFT] Loading point cloud from /content/datasets/nantah-memorial-P42/output/point_clouds/point_cloud_SIFT.ply...
[SIFT] Running outlier removal...
[SIFT] Filtered 8688 extreme outliers. Remaining: 43371 pts.
[SIFT] Interactive 3D plot exported to: /content/datasets/nantah-memorial-P42/output/visualizations/point_cloud_SIFT_visualization.html


##### **SuperPoint**

In [20]:
visualize_point_cloud(
    dataset_name=dataset_name,
    matcher="SuperPoint",
    nb_neighbors=50,
    std_ratio=1.0,
    nb_points=50,
    radius=0.5
)

[SuperPoint] Loading point cloud from /content/datasets/nantah-memorial-P42/output/point_clouds/point_cloud_SuperPoint.ply...
[SuperPoint] Running outlier removal...
[SuperPoint] Filtered 9178 extreme outliers. Remaining: 32660 pts.
[SuperPoint] Interactive 3D plot exported to: /content/datasets/nantah-memorial-P42/output/visualizations/point_cloud_SuperPoint_visualization.html


##### **ORB**

In [21]:
visualize_point_cloud(
    dataset_name=dataset_name,
    matcher="ORB",
    nb_neighbors=50,
    std_ratio=1.0,
    nb_points=100,
    radius=1.0
)

[ORB] Loading point cloud from /content/datasets/nantah-memorial-P42/output/point_clouds/point_cloud_ORB.ply...
[ORB] Running outlier removal...
[ORB] Filtered 10401 extreme outliers. Remaining: 28469 pts.
[ORB] Interactive 3D plot exported to: /content/datasets/nantah-memorial-P42/output/visualizations/point_cloud_ORB_visualization.html
